<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/15_optimization/038_Classification_Optimization-Rasterization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 분류 최적화<br>Classification Optimization


이 노트북은 [`030_Classification_Optimization.ipynb`](030_Classification_Optimization.ipynb) 의 분류 최적화 설정 (두 2D 가우시안 분포의 분류) 에 **히스토그램 기반 데이터 압축** 을 적용한다. 030 은 네 가지 비용 함수 (linear → step → sigmoid → cross-entropy) 의 진화를 다루므로, 여기에서는 마지막 형태인 cross-entropy 만 사용하여 핵심 아이디어 (개별 데이터 vs 격자화된 데이터에서의 최적화 비교) 에 집중한다.<br>
This notebook extends the classification-optimization setup of [`030_Classification_Optimization.ipynb`](030_Classification_Optimization.ipynb) (binary classification of two 2D Gaussians) with **histogram-based data compression**. Since 030 already walks through the cost-function evolution (linear → step → sigmoid → cross-entropy), here we use only the final form (cross-entropy) and focus on the core idea: optimization on raw vs rasterized data.


In [ ]:
import functools
import os
import time
from typing import Dict, List, Tuple, Union


import matplotlib.pyplot as plt
import numpy as np
import numpy.random as nr
import pandas as pd
import scipy.optimize as so


In [ ]:
# to save CI time

if os.getenv('CI', False):
    options = {'maxiter': 10}
else:
    options = None


$(x_1, x_2)$ 데이터 집합 두개 생성<br>Generating two data sets



In [ ]:
set_0_bar = (1, 0)
set_1_bar = (0, 1)


In [ ]:
n = 2000
ndim = 2

set_0 = nr.normal(set_0_bar, [1, 1], (n//2, 2))
set_1 = nr.normal(set_1_bar, [1, 1], (n//2, 2))


생성한 두 데이터 집합을 표시<br>
Visualizing the two data sets



In [ ]:
def plot_two_sets(set_a, set_b, set_a_x_bar=set_0_bar, set_b_x_bar=set_1_bar):

    plt.plot(set_a[:, 0], set_a[:, 1], '.', label="y=0", alpha=0.5)
    plt.plot(set_b[:, 0], set_b[:, 1], '+', label="y=1", alpha=0.5)

    plt.plot(set_a_x_bar[0], set_a_x_bar[1], 'kx')
    plt.plot(set_b_x_bar[0], set_b_x_bar[1], 'kx')

    plt.grid(True)
    plt.axis('equal')
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")


In [ ]:
plot_two_sets(set_0, set_1, set_0_bar, set_1_bar)

xlim_data = plt.xlim()
ylim_data = plt.ylim()

plt.legend(loc=0)
plt.show()
plt.close();


## 데이터 준비<br>Prepare data



### Individual population<br>개별 데이터 포인트



Collect all measurements & labels (set number) into one `numpy.ndarray`<br>
모든 측정 값과 소속 집합 이름표 라벨을 하나의 배열 안에 모음



| measurement 측정값 | label 라벨 |
|:----------:|:-----------:|
| $(x_1, x_2)$ | 0 |
| $(x_1, x_2)$ | 1 |



In [ ]:
y0 = np.zeros((len(set_0), 1))
y1 = np.ones((len(set_1), 1))

data_0 = np.concatenate([set_0, y0], axis=1)
data_1 = np.concatenate([set_1, y1], axis=1)

data = np.concatenate([
        data_0,
        data_1
    ], axis=0
)


행과 열의 갯수 확인<br>
Check the number of rows and columns



In [ ]:
data.shape


처음 10개의 data<br>First 10 data points



In [ ]:
data[:10, :]


마지막 10개의 data<br>Last 10 data points



In [ ]:
data[-10:, :]


### Histogram 히스토그램




In [ ]:
HistogramDict = Dict[int, np.ndarray]
EdgeList = List[np.ndarray]
EdgeDict = Dict[int, EdgeList]


def calc_2d_histograms(
    data:np.ndarray,
    n_bins_list:Tuple[int]=(30, 40),
    ranges:Tuple[Tuple[float]]=((-3.0, +3.0), (-4.0, +4.0),),
) -> List[Union[HistogramDict, EdgeList]]:
    '''
    Calculate histograms from the data population
    '''
    n_pop = data.shape[0]
    ndim = data.shape[1] - 1

    if isinstance(n_bins_list, int):
        n_bins_list = (n_bins_list,) * ndim

    # validate input arguments
    assert all(
        map(
            lambda r:len(r) == ndim,
            ranges
        )
    ), f'ranges = {ranges}'

    label_list = np.unique(data[:, -1])

    hist_dict = {}

    edges_dict = {}

    population_dict = {}

    for label in label_list:
        # histogram for i'th class
        data_label = data[data[:,-1] == label,:]
        assert data_label.shape[1] == (ndim+1), (
            f"data_label.shape = {data_label.shape}\nndim = {ndim}"
        )
        population_dict[label] = data_label.shape[0]

        histogram_label, edges_label = np.histogramdd(
            data_label[:, :-1],
            bins=n_bins_list,
            range=ranges,
        )
        hist_dict[label] = histogram_label # in the shape of (n_bins_list[0], n_bins_list[1])
        assert histogram_label.shape == tuple(n_bins_list), (
            f"label = {label}\n"
            f"histogram_label.shape = {histogram_label.shape}\n"
            f"n_bins_list = {n_bins_list}\n"
        )
        edges_dict[label] = edges_label

    assert sum(population_dict.values()) == n_pop, (
        f"n_pop = {n_pop}\n"
        f"spopulation_dict = {population_dict}\n"
    )

    # verify all histograms have the same edges
    for label in label_list[1:]:
        edge_list = edges_dict[label]
        for axis, (edge, edge_first) in enumerate(zip(edge_list, edges_dict[label_list[0]])):
            assert np.allclose(edge, edge_first),(
                f"label = {label}\n"
                f"axis = {axis}\n"
                f"edges_dict[label] = {edges_dict[label]}\n"
                f"edges_dict[label_list[0]] = {edges_dict[label_list[0]]}\n"
            )

    return [hist_dict, edges_dict[label]]


In [ ]:
hist_dict, edges_list = calc_2d_histograms(data)


In [ ]:
def edges_to_nodes(edges_list:EdgeList,) -> np.ndarray:
    '''
    Convert edges to nodes

    Parameters
    ----------
    edges_list : List[axis_0_array, axis_1_array, ...]

    Returns
    -------
    np.ndarray
        shape = (n_nodes, n_axes)
        n_nodes = (n_bins_0 - 1) * (n_bins_1 - 1) * ...
    '''
    ndim = len(edges_list)

    lower_mesh = np.array(
        np.meshgrid(
            *[edge[:-1] for edge in edges_list],
            indexing='ij'
        )
    ).T.reshape(-1, ndim)
    assert lower_mesh.shape[1] == ndim, (
        f"lower_mesh.shape = {lower_mesh.shape}\n"
        f"len(edges_list) = {ndim}\n"
        f"[len(edge)] = {[len(edge) for edge in edges_list]}\n"
    )
    upper_mesh = np.array(
        np.meshgrid(
            *[edge[1:] for edge in edges_list],
            indexing='ij'
        )
    ).T.reshape(-1, ndim)

    node_mesh = (lower_mesh + upper_mesh) * 0.5
    result = np.hstack((node_mesh, lower_mesh, upper_mesh))
    assert result.shape[1] == (ndim * 3), (
        f"result.shape = {result.shape}\n"
        f"ndim = {ndim}\n"
    )

    return result


In [ ]:
def test_edges_to_nodes():
    edges_list = [
        np.array([0, 1, 2, 3]),
        np.array([-3, -2, -1,]),
    ]

    ndim = 2

    nodes = edges_to_nodes(edges_list)

    assert nodes.shape == (6, ndim * 3), (
        f"nodes.shape = {nodes.shape}\n"
        f"nodes = {nodes}\n"
    )

    expected_nodes = np.array([
        [0.5, -2.5, 0.0, -3.0, 1.0, -2.0],
        [1.5, -2.5, 1.0, -3.0, 2.0, -2.0],
        [2.5, -2.5, 2.0, -3.0, 3.0, -2.0],
        [0.5, -1.5, 0.0, -2.0, 1.0, -1.0],
        [1.5, -1.5, 1.0, -2.0, 2.0, -1.0],
        [2.5, -1.5, 2.0, -2.0, 3.0, -1.0],
    ])   # nodes    # lower    #upper

    assert np.allclose(nodes, expected_nodes), (
        f"nodes = \n{nodes}\n"
        f"expected_nodes = \n{expected_nodes}\n"
    )

test_edges_to_nodes()


In [ ]:
def nodes_and_weights_from_histogram(
    hist_dict:HistogramDict,
    edge_list:EdgeList,
) -> np.ndarray:
    '''
    Output data format (2D case) :
      x  y x_lower y_lower weight label
    x y can be centerpoint of the bin.
    If bin size is small enough, one corner point can be used instead.
    '''

    ndim = len(edge_list)

    nodes = edges_to_nodes(edge_list)

    assert nodes.shape[1] == (ndim * 3), (
        f"nodes.shape = {nodes.shape}\n"
        f"ndim * 3 = {ndim * 3}\n"
    )

    result_list = []

    # add histograms as weight columns
    for k, weight in hist_dict.items():
        assert nodes.shape[0] == weight.size, (
            f"nodes.shape = {nodes.shape}\n"
            f"weights.size = {weight.size}\n"
        )

        weight_column = weight.reshape(-1, 1)
        result_list.append(
            np.hstack(
                (
                    nodes, weight_column, np.full((nodes.shape[0], 1), k)
                )
            )
        )

    result = np.vstack(result_list)

    # remove the rows with weights == 0
    result = result[result[:, -2] > 0]

    return result


## 모델과 비용 함수<br>Model and cost function

030 에서와 동일하게 선형 모델 $\hat y = w_1 x_1 + x_2 + w_2$ 와 cross-entropy 손실을 사용한다. 이 노트북이 Colab 에서 단독 실행될 수 있도록 정의를 다시 모은다.<br>
Same linear model $\hat y = w_1 x_1 + x_2 + w_2$ and cross-entropy loss as in 030. Definitions are gathered here so this notebook can run standalone in Colab.


In [ ]:
def wx(w:np.ndarray, x_y:np.ndarray) -> np.ndarray:
    w1 = w[0]
    w2 = w[1]

    x1 = x_y[:, 0]
    x2 = x_y[:, 1]

    return w1 * x1 + x2 + w2


In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


In [ ]:
def cost_function_cross_entropy(w:np.ndarray, x_y:np.ndarray) -> float:
    n = len(x_y)
    y_hat = sigmoid(wx(w, x_y))
    y = x_y[:, -1]

    cost = -y * np.log2(y_hat) - (1 - y) * np.log2(1 - y_hat)

    return np.mean(cost)


In [ ]:
def get_callback(weight_list=[], cost_list=[]):
  def callback(intermediate_result:so.OptimizeResult):
    assert isinstance(intermediate_result, so.OptimizeResult), (
        f"expected type : {so.OptimizeResult}\n"
        f"passed type : {type(intermediate_result)}\n"
    )
    weight_list.append(intermediate_result.x)
    cost_list.append(intermediate_result.fun)

  return callback


In [ ]:
def plot_decision_boundary(x_min, x_max, weights_array):
    x1_array = np.linspace(x_min, x_max)
    x2_array = - weights_array[0] * x1_array - weights_array[1] + 0.5

    plt.plot(x1_array, x2_array, label="$\hat y = 0.5$")


## 베이스라인: 개별 데이터 최적화<br>Baseline: optimize on raw data

비교 기준으로 cross-entropy 손실을 $n$ 개 개별 데이터 포인트에 대해 직접 최적화한다.<br>
As a baseline, optimize the cross-entropy loss directly over the $n$ individual data points.


In [ ]:
weights_list_cross_entropy = []
cost_list_cross_entropy = []

t_start = time.perf_counter()
result_individual_xent = so.minimize(
    cost_function_cross_entropy,
    x0=np.array([-5.0, 10.0]),
    args=(data,),
    method="Nelder-Mead",
    callback=get_callback(weights_list_cross_entropy, cost_list_cross_entropy),
    options=options,
)
t_individual_xent = time.perf_counter() - t_start

weights = result_individual_xent.x
cost_value = result_individual_xent.fun
n_iter = result_individual_xent.nit
n_call = result_individual_xent.nfev
warning = result_individual_xent.message

result_individual_xent


In [ ]:
plot_two_sets(set_0, set_1)
plot_decision_boundary(data[:, 0].min(), data[:, 0].max(), weights)

plt.legend(loc=0)

plt.show()
plt.close();


## 히스토그램 기반 데이터 압축을 이용한 최적화<br>Optimization using Histogram-based Data Compression


개별 데이터 포인트 $n$ 개에 대해 비용함수를 평가하는 대신, 히스토그램 격자의 $m$ 개 셀 중심점에서 가중치를 적용하여 평가할 수 있다. ($m \ll n$)<br>
Instead of evaluating the cost function on $n$ individual data points, we can evaluate at $m$ histogram cell centroids with weights. ($m \ll n$)


이는 PCX 파일의 런-렝스 인코딩과 유사한 개념이다: 같은 구간에 속하는 데이터 포인트들을 (개수, 중심점) 쌍으로 압축한다.<br>
This is analogous to run-length encoding in PCX files: data points in the same bin are compressed into (count, centroid) pairs.


이 압축 기법은 [Lee, 2004](#references-참고문헌) Ch. 6 에서 도입되었으며, 두 개의 5차원 히스토그램으로 3.4M 자동차 주행 데이터 포인트 (약 400 MB) 를 약 1.9 MB 로 압축하여 충돌 경고 알고리즘 매개변수 최적화에 사용되었다.<br>
This compression technique was introduced in [Lee, 2004](#references-참고문헌) Ch. 6, where two 5D histograms reduced 3.4 M automotive driving-data points (approx. 400 MB) to approx. 1.9 MB for parameter optimization of collision-warning algorithms.


### 가중 비용 함수<br>Weighted Cost Function

$$
C_w = \frac{\sum_{i=1}^{m} w_i \cdot \text{loss}_i}{\sum_{i=1}^{m} w_i}
$$


### 격자화된 데이터 준비<br>Prepare Rasterized Data


In [ ]:
raster_data = nodes_and_weights_from_histogram(hist_dict, edges_list)
print(f"Individual data points 개별 데이터 수: {data.shape[0]}")
print(f"Rasterized data points 격자화 데이터 수: {raster_data.shape[0]}")
print(f"Compression ratio 압축률: {data.shape[0] / raster_data.shape[0]:.1f}x")


### 가중 비용 함수 정의<br>Define Weighted Cost Functions

압축된 데이터의 가중치를 반영하여 비용함수를 계산한다.<br>
Cost functions are computed using the weights from the compressed data.


In [ ]:
def weighted_cost_cross_entropy(w, x_y):
    weights = x_y[:, -2]
    y_hat = sigmoid(wx(w, x_y))
    y = x_y[:, -1]
    cost = -y * np.log2(y_hat) - (1 - y) * np.log2(1 - y_hat)
    return np.sum(weights * cost) / np.sum(weights)


### 격자화된 데이터로 최적화 실행<br>Run Optimization with Rasterized Data


In [ ]:
weight_list_raster_xent = []
cost_list_raster_xent = []

t_start = time.perf_counter()
result_raster_xent = so.minimize(
    weighted_cost_cross_entropy,
    x0=np.array([-5.0, 10.0]),
    args=(raster_data,),
    method="Nelder-Mead",
    callback=get_callback(weight_list_raster_xent, cost_list_raster_xent),
    options=options,
)
t_raster_xent = time.perf_counter() - t_start

print(f"Rasterized cross entropy 격자화 교차 엔트로피: {t_raster_xent:.4f}s, nfev={result_raster_xent.nfev}")
result_raster_xent


### 격자화 데이터 수렴 그래프<br>Convergence Plot for Rasterized Data


In [ ]:
plt.plot(cost_list_cross_entropy, '.-', label='individual (raw data)')
plt.plot(cost_list_raster_xent, '.-', label='rasterized (histogram)')
plt.xlabel('iteration')
plt.ylabel('cross-entropy cost')
plt.legend(loc=0)
plt.grid(True)
plt.title('Convergence: raw vs rasterized')
plt.show()
plt.close();


### 결정 경계 비교: 개별 데이터 vs 격자화 데이터<br>Decision Boundary Comparison: Individual vs Rasterized Data


In [ ]:
plot_two_sets(set_0, set_1)

# Individual cross-entropy boundary
x1_arr = np.linspace(data[:, 0].min(), data[:, 0].max())
x2_individual = -result_individual_xent.x[0] * x1_arr - result_individual_xent.x[1] + 0.5
plt.plot(x1_arr, x2_individual, 'r-', linewidth=2, label='Individual cross-entropy')

# Rasterized cross-entropy boundary
x2_raster = -result_raster_xent.x[0] * x1_arr - result_raster_xent.x[1] + 0.5
plt.plot(x1_arr, x2_raster, 'g--', linewidth=2, label='Rasterized cross-entropy')

plt.legend(loc=0)
plt.title('Decision Boundary Comparison')
plt.show()
plt.close();


### 비용 함수 표면 비교<br>Cost-function surface comparison

두 비용 함수의 윤곽선을 매개변수 평면 ($w_1$, $w_2$) 위에서 나란히 비교한다. 격자화는 개별 데이터의 비용 함수를 이산화한 근사이므로, 두 표면은 거의 동일해야 하고 전역 최소점도 같은 위치 부근에 나타나야 한다. 각 최적화의 진행 경로 (callback 으로 기록) 가 그 위에 표시된다.<br>
Compare the two cost functions' contours side by side over the parameter plane ($w_1$, $w_2$). Rasterization is a discretized approximation of the individual cost function, so the two surfaces should be nearly identical and the global minimum should sit at approximately the same location. The optimizer paths (recorded via callback) are overlaid.

이는 [Lee, 2004](#references-참고문헌) Ch. 6 Fig. 6.3 의 윤곽선+경사 시각화와 같은 종류의 그림이다.<br>
This mirrors Fig. 6.3 in [Lee, 2004](#references-참고문헌) Ch. 6 (contour and gradient field over the parameter plane).


In [ ]:
def calc_cost_surf(cost_function, x_y, w_range=(-10.0, 10.0), n=41):
    '''Evaluate cost_function over an n x n grid in (w1, w2) space.'''
    w1 = np.linspace(*w_range, n)
    w2 = np.linspace(*w_range, n)
    W1, W2 = np.meshgrid(w1, w2)
    C = np.zeros_like(W1)
    for i_row in range(n):
        for j_col in range(n):
            w = np.array([W1[i_row, j_col], W2[i_row, j_col]])
            C[i_row, j_col] = cost_function(w, x_y)
    return W1, W2, C


In [ ]:
# Both surfaces on the same (w1, w2) grid
W1, W2, C_indiv = calc_cost_surf(cost_function_cross_entropy, data)
_, _, C_raster = calc_cost_surf(weighted_cost_cross_entropy, raster_data)

# Common contour levels for a fair side-by-side
levels = np.linspace(min(C_indiv.min(), C_raster.min()),
                     max(C_indiv.max(), C_raster.max()), 15)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

weights_indiv  = np.array(weights_list_cross_entropy)
weights_raster = np.array(weight_list_raster_xent)

axes[0].contour(W1, W2, C_indiv, levels=levels, cmap='viridis')
axes[0].plot(weights_indiv[:, 0], weights_indiv[:, 1], 'r.-', alpha=0.7, label='optimizer path')
axes[0].plot(result_individual_xent.x[0], result_individual_xent.x[1], 'r*', markersize=15, label='optimum')
axes[0].set_title(f'Individual (n={data.shape[0]} points)')
axes[0].set_xlabel(r'$w_1$')
axes[0].set_ylabel(r'$w_2$')
axes[0].grid(True)
axes[0].legend(loc=0)

axes[1].contour(W1, W2, C_raster, levels=levels, cmap='viridis')
axes[1].plot(weights_raster[:, 0], weights_raster[:, 1], 'g.-', alpha=0.7, label='optimizer path')
axes[1].plot(result_raster_xent.x[0], result_raster_xent.x[1], 'g*', markersize=15, label='optimum')
axes[1].set_title(f'Rasterized (n={raster_data.shape[0]} cells)')
axes[1].set_xlabel(r'$w_1$')
axes[1].grid(True)
axes[1].legend(loc=0)

fig.suptitle('Cost-function surface: individual vs rasterized')
plt.tight_layout()
plt.show()
plt.close();


### 점별 평가<br>Point-by-point evaluation

주의: `result_raster_xent.fun` 은 격자화된 데이터 (히스토그램 근사값) 에 대한 최종 비용이지, 원래 데이터에 대한 비용이 아니다. 따라서 `result_individual_xent.fun` (개별 데이터에 대한 비용) 과 직접 비교할 수 없다. 공정한 비교를 위해 격자화 최적화로 얻은 모수를 **원래 데이터 전체에 대해 점별로 평가** 한다.<br>
Note: `result_raster_xent.fun` is the final cost on the rasterized data (the histogram approximation), not on the raw data. It is not directly comparable to `result_individual_xent.fun` (cost on individual data). For a fair comparison, evaluate the parameters from the rasterized optimization on the **full raw data, point by point**.


이는 [Lee, 2004](#references-참고문헌) Ch. 6 §6.1 의 패턴을 따른다: 격자화된 히스토그램에서 최적화하여 매개변수를 얻은 뒤, 최종 평가는 원래 데이터에 대해 점별로 수행한다.<br>
This follows the pattern in [Lee, 2004](#references-참고문헌) Ch. 6 §6.1: optimize on the rasterized histogram to obtain parameters, then perform the final evaluation point by point on the raw data.


In [ ]:
# Point-by-point evaluation: raster-optimized params, evaluated on the full data
cost_raster_params_on_raw = cost_function_cross_entropy(result_raster_xent.x, data)

print(f'Individual params, individual cost (point-by-point):  {result_individual_xent.fun:.6f}')
print(f'Rasterized params, rasterized cost (on histogram):    {result_raster_xent.fun:.6f}')
print(f'Rasterized params, individual cost (point-by-point):  {cost_raster_params_on_raw:.6f}  ← validation')


### 결과 비교표<br>Comparison Table


In [ ]:
comparison = pd.DataFrame({
    'Method 방법': [
        'Individual cross-entropy 개별 교차엔트로피',
        'Rasterized cross-entropy 격자화 교차엔트로피',
    ],
    'Data points 데이터 수': [
        data.shape[0],
        raster_data.shape[0],
    ],
    'Compression 압축률': [
        '1x',
        f'{data.shape[0] / raster_data.shape[0]:.1f}x',
    ],
    'nfev': [
        result_individual_xent.nfev,
        result_raster_xent.nfev,
    ],
    'time (s) 시간': [
        f'{t_individual_xent:.4f}',
        f'{t_raster_xent:.4f}',
    ],
    'cost during optim 최적화 중 비용': [
        f'{result_individual_xent.fun:.6f}',
        f'{result_raster_xent.fun:.6f}  (on histogram)',
    ],
    'cost point-by-point 점별 평가 비용': [
        f'{result_individual_xent.fun:.6f}',
        f'{cost_raster_params_on_raw:.6f}',
    ],
    'w1': [
        f'{result_individual_xent.x[0]:.4f}',
        f'{result_raster_xent.x[0]:.4f}',
    ],
    'w2': [
        f'{result_individual_xent.x[1]:.4f}',
        f'{result_raster_xent.x[1]:.4f}',
    ],
})
comparison


## Final Bell<br>마지막 종



In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");


## References<br>참고문헌
* Lee, K. (2004). *Longitudinal Driver Model and Collision Warning and Avoidance Algorithms Based on Human Driving Databases.* Ph.D. dissertation, Mechanical Engineering, University of Michigan.
* J. Santarcangelo, Deep Neural Networks with PyTorch, Coursera
* S. Kim, Deep Learning for Everyone, http://hunkim.github.io/ml/
* SciPy Developer Community, Optimization and root finding, SciPy Documentation, https://docs.scipy.org/doc/scipy/reference/optimize.html
